<a href="https://colab.research.google.com/github/MP3006/Multi-Asset-Portfolio-Risk-and-Performance-Report/blob/main/Multi%20Asset%20Portfolio%20Risk%20Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Multi-Asset Portfolio Risk and Performance Report.

The script downloads market data, calculates portfolio and asset-level KPIs,
creates visual analytics, and saves all results to an outputs directory.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf


OUTPUT_DIR = Path.cwd() / "outputs"
ASSETS = {
    "SPY": "US Equities",
    "QQQ": "Technology Equities",
    "TLT": "Long-Term Treasuries",
    "GLD": "Gold",
    "BTC-USD": "Bitcoin",
}
WEIGHTS = pd.Series(
    {
        "US Equities": 0.30,
        "Technology Equities": 0.20,
        "Long-Term Treasuries": 0.20,
        "Gold": 0.15,
        "Bitcoin": 0.15,
    }
)
TRADING_DAYS = 252
RISK_FREE_RATE = 0.02
VAR_CONFIDENCE = 0.95
ROLLING_WINDOW = 63


def validate_configuration():
    """Validate portfolio weights before downloading data."""
    if set(WEIGHTS.index) != set(ASSETS.values()):
        raise ValueError("Every asset must have exactly one portfolio weight.")
    if (WEIGHTS < 0).any():
        raise ValueError("Portfolio weights cannot be negative.")
    if not np.isclose(WEIGHTS.sum(), 1.0):
        raise ValueError("Portfolio weights must sum to 1.0.")


def download_prices():
    """Download five years of adjusted closing prices from Yahoo Finance."""
    raw = yf.download(
        list(ASSETS),
        period="5y",
        auto_adjust=True,
        progress=False,
        threads=False,
        timeout=20,
    )
    if raw.empty:
        raise ValueError("Yahoo Finance returned an empty dataset.")

    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw["Close"].copy()
    else:
        prices = raw[["Close"]].copy()

    missing = [ticker for ticker in ASSETS if ticker not in prices.columns]
    if missing:
        raise ValueError(f"Missing price data for: {', '.join(missing)}")

    prices = prices[list(ASSETS)].rename(columns=ASSETS)
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    prices = prices[prices.index.dayofweek < 5].ffill().dropna()
    if len(prices) < ROLLING_WINDOW + 20:
        raise ValueError("Insufficient observations for rolling analysis.")
    return prices


def generate_fallback_prices():
    """Generate reproducible demonstration prices if download is unavailable."""
    generator = np.random.default_rng(42)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=1260)
    names = list(ASSETS.values())

    annual_returns = np.array([0.09, 0.11, 0.035, 0.055, 0.16])
    annual_vols = np.array([0.18, 0.24, 0.14, 0.16, 0.55])
    correlation = np.array(
        [
            [1.00, 0.88, -0.20, 0.05, 0.35],
            [0.88, 1.00, -0.18, 0.03, 0.40],
            [-0.20, -0.18, 1.00, 0.20, -0.05],
            [0.05, 0.03, 0.20, 1.00, 0.10],
            [0.35, 0.40, -0.05, 0.10, 1.00],
        ]
    )
    covariance = np.outer(annual_vols, annual_vols) * correlation / TRADING_DAYS
    daily_means = annual_returns / TRADING_DAYS
    simulated_returns = generator.multivariate_normal(
        daily_means, covariance, size=len(dates)
    )
    start_values = np.array([400.0, 330.0, 140.0, 180.0, 30000.0])
    prices = start_values * np.exp(np.cumsum(simulated_returns, axis=0))
    return pd.DataFrame(prices, index=dates, columns=names)


def calculate_drawdown(return_series):
    """Calculate drawdown from a daily return series."""
    wealth = (1 + return_series).cumprod()
    running_peak = wealth.cummax()
    return wealth / running_peak - 1


def performance_metrics(return_series):
    """Calculate commonly used return, risk, and downside-risk measures."""
    clean = return_series.dropna()
    if len(clean) < 2:
        raise ValueError("At least two returns are required for performance metrics.")

    wealth = (1 + clean).cumprod()
    total_return = wealth.iloc[-1] - 1
    years = len(clean) / TRADING_DAYS
    annual_return = wealth.iloc[-1] ** (1 / years) - 1
    annual_volatility = clean.std(ddof=1) * np.sqrt(TRADING_DAYS)

    downside_returns = clean[clean < 0]
    downside_deviation = downside_returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = (
        (annual_return - RISK_FREE_RATE) / annual_volatility
        if annual_volatility > 0
        else np.nan
    )
    sortino = (
        (annual_return - RISK_FREE_RATE) / downside_deviation
        if downside_deviation > 0
        else np.nan
    )

    drawdown = calculate_drawdown(clean)
    maximum_drawdown = drawdown.min()
    calmar = (
        annual_return / abs(maximum_drawdown)
        if maximum_drawdown < 0
        else np.nan
    )

    var_level = np.quantile(clean, 1 - VAR_CONFIDENCE)
    tail_losses = clean[clean <= var_level]
    cvar_level = tail_losses.mean() if not tail_losses.empty else np.nan

    return {
        "Total Return (%)": total_return * 100,
        "Annual Return (%)": annual_return * 100,
        "Annual Volatility (%)": annual_volatility * 100,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Maximum Drawdown (%)": maximum_drawdown * 100,
        "Calmar Ratio": calmar,
        "Daily VaR 95% (%)": var_level * 100,
        "Daily CVaR 95% (%)": cvar_level * 100,
    }


def build_performance_table(asset_returns, portfolio_returns):
    """Calculate metrics for every asset and the combined portfolio."""
    results = {
        asset: performance_metrics(asset_returns[asset])
        for asset in asset_returns.columns
    }
    results["Portfolio"] = performance_metrics(portfolio_returns)
    table = pd.DataFrame(results).T
    table.index.name = "Asset"
    return table.round(3)


def calculate_risk_contribution(asset_returns):
    """Calculate each holding's contribution to annual portfolio volatility."""
    covariance = asset_returns.cov() * TRADING_DAYS
    ordered_weights = WEIGHTS.reindex(asset_returns.columns)
    portfolio_variance = float(ordered_weights @ covariance @ ordered_weights)
    portfolio_volatility = np.sqrt(portfolio_variance)
    marginal_contribution = covariance @ ordered_weights / portfolio_volatility
    component_contribution = ordered_weights * marginal_contribution
    contribution_percent = component_contribution / portfolio_volatility * 100

    table = pd.DataFrame(
        {
            "Portfolio Weight (%)": ordered_weights * 100,
            "Annual Risk Contribution (%)": component_contribution * 100,
            "Share of Portfolio Risk (%)": contribution_percent,
        }
    )
    table.index.name = "Asset"
    return table.round(3)


def save_tables(prices, returns, portfolio_returns, performance, risk_table):
    """Save clean datasets and analytical tables as CSV files."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prices.to_csv(OUTPUT_DIR / "adjusted_prices.csv")
    returns.to_csv(OUTPUT_DIR / "asset_returns.csv")
    performance.to_csv(OUTPUT_DIR / "performance_summary.csv")
    risk_table.to_csv(OUTPUT_DIR / "risk_contribution.csv")
    returns.corr().round(4).to_csv(OUTPUT_DIR / "correlation_matrix.csv")
    portfolio_returns.rename("Portfolio Return").to_csv(
        OUTPUT_DIR / "portfolio_returns.csv"
    )


def plot_cumulative_returns(asset_returns, portfolio_returns):
    """Plot the growth of 100 for all assets and the portfolio."""
    combined = asset_returns.copy()
    combined["Portfolio"] = portfolio_returns
    wealth = (1 + combined).cumprod() * 100
    ax = wealth.plot(figsize=(12, 6), linewidth=1.5)
    ax.set_title("Growth of 100: Assets and Portfolio")
    ax.set_ylabel("Portfolio value")
    ax.set_xlabel("")
    ax.grid(linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "01_cumulative_returns.png", dpi=160)
    plt.close()


def plot_drawdowns(asset_returns, portfolio_returns):
    """Plot historical drawdowns for the assets and portfolio."""
    combined = asset_returns.copy()
    combined["Portfolio"] = portfolio_returns
    drawdowns = combined.apply(calculate_drawdown) * 100
    ax = drawdowns.plot(figsize=(12, 6), linewidth=1.1)
    ax.set_title("Drawdown Analysis")
    ax.set_ylabel("Drawdown (%)")
    ax.set_xlabel("")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.grid(linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "02_drawdowns.png", dpi=160)
    plt.close()


def plot_rolling_volatility(asset_returns, portfolio_returns):
    """Plot rolling annualised volatility over a three-month window."""
    combined = asset_returns.copy()
    combined["Portfolio"] = portfolio_returns
    rolling_volatility = combined.rolling(ROLLING_WINDOW).std() * np.sqrt(
        TRADING_DAYS
    ) * 100
    ax = rolling_volatility.plot(figsize=(12, 6), linewidth=1.2)
    ax.set_title(f"Rolling {ROLLING_WINDOW}-Day Annualised Volatility")
    ax.set_ylabel("Volatility (%)")
    ax.set_xlabel("")
    ax.grid(linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_rolling_volatility.png", dpi=160)
    plt.close()


def plot_correlation_heatmap(asset_returns):
    """Create a labelled correlation heatmap using Matplotlib."""
    correlation = asset_returns.corr()
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(correlation, cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=35)
    ax.set_yticks(range(len(correlation.index)), correlation.index)
    for row in range(len(correlation.index)):
        for column in range(len(correlation.columns)):
            ax.text(
                column,
                row,
                f"{correlation.iloc[row, column]:.2f}",
                ha="center",
                va="center",
                fontsize=9,
            )
    ax.set_title("Asset Return Correlation")
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "04_correlation_heatmap.png", dpi=160)
    plt.close()


def plot_risk_contribution(risk_table):
    """Compare capital weights with shares of portfolio risk."""
    chart_data = risk_table[
        ["Portfolio Weight (%)", "Share of Portfolio Risk (%)"]
    ]
    ax = chart_data.plot(kind="bar", figsize=(10, 5), color=["#4c78a8", "#e45756"])
    ax.set_title("Portfolio Weight vs Share of Portfolio Risk")
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "05_risk_contribution.png", dpi=160)
    plt.close()


def print_report(performance, risk_table, prices, source):
    """Print the most useful results to the terminal."""
    print("\nMULTI-ASSET PORTFOLIO RISK AND PERFORMANCE REPORT")
    print("=" * 72)
    print(f"Data source: {source}")
    print(f"Analysis period: {prices.index.min().date()} to {prices.index.max().date()}")
    print("\nPERFORMANCE AND RISK KPIs")
    print(performance.to_string())
    print("\nPORTFOLIO RISK CONTRIBUTION")
    print(risk_table.to_string())


def main():
    """Run the complete analytics workflow."""
    try:
        validate_configuration()
        try:
            prices = download_prices()
            source = "Yahoo Finance"
        except Exception as download_error:
            print(f"Market download unavailable: {download_error}")
            print("Using reproducible demonstration prices instead.")
            prices = generate_fallback_prices()
            source = "Generated demonstration data"

        asset_returns = prices.pct_change().dropna()
        ordered_weights = WEIGHTS.reindex(asset_returns.columns)
        portfolio_returns = asset_returns.mul(ordered_weights, axis=1).sum(axis=1)

        performance = build_performance_table(asset_returns, portfolio_returns)
        risk_table = calculate_risk_contribution(asset_returns)

        save_tables(prices, asset_returns, portfolio_returns, performance, risk_table)
        plot_cumulative_returns(asset_returns, portfolio_returns)
        plot_drawdowns(asset_returns, portfolio_returns)
        plot_rolling_volatility(asset_returns, portfolio_returns)
        plot_correlation_heatmap(asset_returns)
        plot_risk_contribution(risk_table)
        print_report(performance, risk_table, prices, source)
        print(f"\nOutputs saved to: {OUTPUT_DIR.resolve()}")
    except (ValueError, KeyError, TypeError, ZeroDivisionError) as error:
        print(f"Analysis could not be completed: {error}")
    except Exception as error:
        print(f"Unexpected error: {error}")


if __name__ == "__main__":
    main()


MULTI-ASSET PORTFOLIO RISK AND PERFORMANCE REPORT
Data source: Yahoo Finance
Analysis period: 2021-06-21 to 2026-06-19

PERFORMANCE AND RISK KPIs
                      Total Return (%)  Annual Return (%)  Annual Volatility (%)  Sharpe Ratio  Sortino Ratio  Maximum Drawdown (%)  Calmar Ratio  Daily VaR 95% (%)  Daily CVaR 95% (%)
Asset                                                                                                                                                                                   
US Equities                     89.842             13.188                 16.810         0.666          0.899               -24.496         0.538             -1.641              -2.430
Technology Equities            121.368             16.599                 22.203         0.658          0.911               -35.119         0.473             -2.219              -3.194
Long-Term Treasuries           -28.338             -6.236                 15.506        -0.531         -0.838    